In [1]:
# %%
import numpy as np

NON1_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_non-speech_1.npy"
NON2_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_non-speech_2.npy"
SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_speech.npy"

X_non1 = np.load(NON1_PATH, mmap_mode="r")
X_non2 = np.load(NON2_PATH, mmap_mode="r")
X_speech = np.load(SPEECH_PATH, mmap_mode="r")

print("Non1 :", X_non1.shape)
print("Non2 :", X_non2.shape)
print("Speech:", X_speech.shape)

Non1 : (1094415, 60)
Non2 : (168465, 60)
Speech: (1253962, 60)


In [2]:
# %%
# %%
X_non = np.concatenate([
    X_non1[:, :60],
    X_non2[:, :60]
], axis=0)

X_speech = X_speech[:, :60]

print("Non total:", X_non.shape)
print("Speech:", X_speech.shape)

X = np.vstack([X_non, X_speech]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

print("X:", X.shape)
print("Label distribution:", np.bincount(y))

Non total: (1262880, 60)
Speech: (1253962, 60)
X: (2516842, 60)
Label distribution: [1262880 1253962]


In [3]:
# %%
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2013473, 60)
Test : (503369, 60)


In [4]:
# %%
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

rf = RandomForestClassifier(
    n_estimators=80,
    max_depth=15,
    min_samples_leaf=10,
    n_jobs=4,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

=== RANDOM FOREST ===
Accuracy : 0.9438920553311785
Precision: 0.9376115904639803
Recall   : 0.95064056811793
F1-score : 0.9440811291877775
AUC      : 0.9866281536743606


In [5]:
# %%
from xgboost import XGBClassifier

xgb = XGBClassifier(
    tree_method="hist",
    device="cuda",          # dùng GPU
    max_depth=10,
    n_estimators=300,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBOOST (GPU) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

/home/quochuy/miniconda3/envs/audio_ml/lib/python3.10/site-packages/xgboost/core.py:774: UserWarning: [22:19:39] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


=== XGBOOST (GPU) ===
Accuracy : 0.9702881981210603
Precision: 0.9695273210879858
Recall   : 0.9708803674743713
F1-score : 0.9702033725415192
AUC      : 0.9961675551988697


In [6]:
# %%
import joblib

# Lưu Random Forest
joblib.dump(rf, "rf_lfcc_60feat.pkl")

# Lưu XGBoost
joblib.dump(xgb, "xgb_lfcc_60feat_gpu.pkl")

print("Models saved successfully!")

Models saved successfully!
